In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from Temis.dataloader import German

# Common Auxiliary Libraries
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

# Uncommon Auxiliary Libraries
import jax 
import jax.numpy as jnp

# Machine Learning Models Libraries
from Temis.models import LogisticRegression as FairLR
from sklearn.linear_model import LogisticRegression as SKLearnLR

# Machine Learning Metrics Libraries
from sklearn.metrics import roc_auc_score
from Temis.comparison_utils.cmp_fairness import extract_fairness, debugger_woogle

In [3]:
german_preprocessed_path = os.path.join('..', 'data', 'preprocessed', 'german.csv')
german = German(german_preprocessed_path)
X_train, y_train = german.get_train_data()
X_test, y_test = german.get_test_data()
print(X_train['ForeignWorker_A202'])

729    0.0
206    0.0
314    1.0
522    0.0
711    0.0
      ... 
989    0.0
642    0.0
458    0.0
559    0.0
349    0.0
Name: ForeignWorker_A202, Length: 750, dtype: float64


In [4]:
s_attribute = 'ForeignWorker_A202'

In [5]:
model_fair = FairLR(epochs=100, penalty='l2', penalty_weight=1.0, fair_penalty='Rpr', fair_penalty_weight=10.0)
model_fair.fit(X_train, y_train, S=X_train['ForeignWorker_A202'], debug=False)
fair_pred_test = model_fair.predict(X_test)
fair_prob_test = model_fair.predict_probability(X_test)

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


In [9]:
# By default uses 'l2' penalty
model_sk = SKLearnLR(max_iter=100, solver='lbfgs')
model_sk.fit(X_train, y_train)
sk_pred_test = model_sk.predict(X_test)
sk_prob_test = model_sk.predict(X_test)

In [10]:
print(f'AUC: {roc_auc_score(y_test, fair_prob_test)}')
print(f'AUC: {roc_auc_score(y_test, sk_prob_test)}')

AUC: 0.6806748703660708
AUC: 0.6880272424734928


In [11]:
print(X_test[s_attribute])

496    0.0
936    0.0
377    0.0
86     0.0
576    0.0
      ... 
787    0.0
847    0.0
570    0.0
47     0.0
363    0.0
Name: ForeignWorker_A202, Length: 250, dtype: float64


In [15]:
fair_fairness = extract_fairness(model_fair, X_test, y_test, s_attribute)

Initiating extract_fairness execution...
496    0.0
936    0.0
377    0.0
86     0.0
576    0.0
      ... 
787    0.0
847    0.0
570    0.0
47     0.0
363    0.0
Name: ForeignWorker_A202, Length: 250, dtype: float64


IndexError: only integers, slices (`:`), ellipsis (`...`), newaxis (`None`) and integer or boolean arrays are valid indices. Got 496    False
936    False
377    False
86     False
576    False
       ...  
787    False
847    False
570    False
47     False
363    False
Name: ForeignWorker_A202, Length: 250, dtype: bool